In [4]:
from oqd_compiler_infrastructure import (
    CFGBlock,
    CFGBlockAccumulator,
    Post,
    PrettyPrint,
    cfg_to_dot,
    gen_pass,
)
from rich import print as pprint

from oqd_core.analysis.analog.cfg import AnalogCFGBuilder, AnalogCFGtoAST
from oqd_core.analysis.analog.dim_checker import DimensionChecker
from oqd_core.analysis.analog.reaching_def import AvailableVariableAnalysis
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker, get_type_name
from oqd_core.analysis.dominator import PostDominatorAnalysis
from oqd_core.frontend.analog import parse_analog, serialize_analog

printer = Post(PrettyPrint())

# source = """
# a = 1
# pi = 3.141592654

# q = qreg(3)
# initialize(q)

# w = [q[0],q[1]]

# H = %X %@ %X %+ %I %@ %X %+ %Y %@ (%Z %+ %X)

# while (a < 10) {
#     evolve(H,1,w)

#     a = a + 1
# }

# m = measure(q)
# """

with open("test.analog", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_analog(source)
cfg = AnalogCFGBuilder()(circuit)
cfg = CFGBlockAccumulator()(cfg)


In [5]:
undef_checker = AvailableVariableAnalysis()
undef_res = undef_checker.analyze(cfg)

type_checker = AnalogTypeChecker()
type_res = type_checker.analyze(cfg)

dim_checker = DimensionChecker()
dim_res = dim_checker.analyze(cfg)


In [6]:
dot = cfg_to_dot(cfg, serialize=serialize_analog, max_lines=1)
dot.render("cfg", format="png", cleanup=True)


'cfg.png'

In [7]:
analyzer = PostDominatorAnalysis()

pdom_result = analyzer.analyze(cfg)

circuit = AnalogCFGtoAST(pdom_result=pdom_result)(cfg)

with open("reserialized_test.analog", "w") as f:
    f.write(serialize_analog(circuit))